# 00 - Train/Test split (stratified by amount bins and win/loss)

This notebook creates a train/test split stratifying on a combined key:
- `Opportunity Result Bool` (Won/Loss target)
- `Opportunity Amount USD` quantile bin (qcut)

Outputs are saved to `data/intermediate/` as parquet files.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = '../../data/intermediate/df_tagged_full.parquet'
TARGET_COL = 'Opportunity Result Bool'
AMOUNT_COL = 'Opportunity Amount USD'

df = pd.read_parquet(DATA_PATH).copy()
print('Loaded:', DATA_PATH)
print('shape:', df.shape)
df[[TARGET_COL, AMOUNT_COL]].head()

Loaded: ../../data/intermediate/df_tagged_full.parquet
shape: (78025, 35)


,Opportunity Result Bool,Opportunity Amount USD
0,True,0
1,False,0
2,True,7750
3,False,0
4,False,69756


In [3]:
# Keep rows with valid target/amount
work = df.dropna(subset=[TARGET_COL, AMOUNT_COL]).copy()

# Build amount quantile bins (fallback to fewer bins if needed)
candidate_q = [10, 8, 6, 5, 4, 3, 2]
selected_q = None
for q in candidate_q:
    try:
        bins = pd.qcut(work[AMOUNT_COL], q=q, duplicates='drop')
        key = work[TARGET_COL].astype(int).astype(str) + '__' + bins.astype(str)
        if key.value_counts().min() >= 2:
            selected_q = q
            work['amount_qbin'] = bins
            work['stratify_key'] = key
            break
    except ValueError:
        continue

if selected_q is None:
    raise ValueError('Could not build a valid stratification key with min class count >= 2')

print('Rows used for split:', len(work))
print('qcut bins selected:', selected_q)
print('stratify classes:', work['stratify_key'].nunique())
print('min class count:', int(work['stratify_key'].value_counts().min()))
work[['amount_qbin', 'stratify_key']].head()

Rows used for split: 78025
qcut bins selected: 10
stratify classes: 20
min class count: 698


,amount_qbin,stratify_key
0,"(-0.001, 5000.0]","1__(-0.001, 5000.0]"
1,"(-0.001, 5000.0]","0__(-0.001, 5000.0]"
2,"(5000.0, 10420.0]","1__(5000.0, 10420.0]"
3,"(-0.001, 5000.0]","0__(-0.001, 5000.0]"
4,"(61000.0, 100000.0]","0__(61000.0, 100000.0]"


In [4]:
TRAIN_SIZE = 0.7
RANDOM_STATE = 42

train_df, test_df = train_test_split(
    work,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_STATE,
    stratify=work['stratify_key']
)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('train ratio:', round(len(train_df) / len(work), 4))
print('test ratio:', round(len(test_df) / len(work), 4))

train shape: (54617, 37)
test shape: (23408, 37)
train ratio: 0.7
test ratio: 0.3


In [5]:
def summarize_split(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    out = (
        frame.groupby('stratify_key', observed=False)
        .size()
        .rename('n')
        .to_frame()
    )
    out['pct'] = out['n'] / out['n'].sum()
    out['split'] = name
    return out.reset_index()

sum_train = summarize_split(train_df, 'train')
sum_test = summarize_split(test_df, 'test')
sum_all = summarize_split(work, 'all')

summary = pd.concat([sum_all, sum_train, sum_test], ignore_index=True)
display(summary.head(15))

pivot = summary.pivot(index='stratify_key', columns='split', values='pct').fillna(0)
pivot['abs_diff_train_vs_all'] = (pivot['train'] - pivot['all']).abs()
pivot['abs_diff_test_vs_all'] = (pivot['test'] - pivot['all']).abs()
print('max abs diff train vs all:', float(pivot['abs_diff_train_vs_all'].max()))
print('max abs diff test  vs all:', float(pivot['abs_diff_test_vs_all'].max()))
display(pivot.sort_values('abs_diff_test_vs_all', ascending=False).head(15))

,stratify_key,n,pct,split
0,"0__(-0.001, 5000.0]",5440,0.069721,all
1,"0__(100000.0, 131000.0]",5199,0.066632,all
2,"0__(10420.0, 20000.0]",7230,0.092663,all
3,"0__(131000.0, 220000.0]",6730,0.086254,all
4,"0__(20000.0, 30000.0]",5496,0.070439,all
5,"0__(220000.0, 1000000.0]",6305,0.080807,all
6,"0__(30000.0, 49000.0]",4292,0.055008,all
7,"0__(49000.0, 61000.0]",6781,0.086908,all
8,"0__(5000.0, 10420.0]",4817,0.061737,all
9,"0__(61000.0, 100000.0]",8108,0.103915,all


max abs diff train vs all: 8.414894323058308e-06
max abs diff test  vs all: 1.963415427386206e-05


split,all,test,train,abs_diff_train_vs_all,abs_diff_test_vs_all
stratify_key,,,,,
"0__(220000.0, 1000000.0]",0.080807,0.080827,0.080799,0.000008,0.000020
"0__(61000.0, 100000.0]",0.103915,0.103896,0.103924,0.000008,0.000019
"1__(49000.0, 61000.0]",0.012791,0.012773,0.012798,0.000007,0.000017
"1__(100000.0, 131000.0]",0.008946,0.008929,0.008953,0.000007,0.000017
"1__(131000.0, 220000.0]",0.013867,0.013884,0.013860,0.000007,0.000017
"0__(30000.0, 49000.0]",0.055008,0.055024,0.055001,0.000007,0.000016
"0__(49000.0, 61000.0]",0.086908,0.086893,0.086914,0.000006,0.000015
"1__(10420.0, 20000.0]",0.036283,0.036270,0.036289,0.000006,0.000014
"1__(20000.0, 30000.0]",0.022416,0.022428,0.022411,0.000005,0.000012


In [6]:
TRAIN_PATH = '../../data/intermediate/df_train_stratified.parquet'
TEST_PATH = '../../data/intermediate/df_test_stratified.parquet'
SUMMARY_PATH = '../../data/intermediate/df_train_test_stratified_summary.csv'

# Keep helper columns in output for traceability (cast interval bins to string for parquet compatibility)
train_out = train_df.copy()
test_out = test_df.copy()
train_out['amount_qbin'] = train_out['amount_qbin'].astype(str)
test_out['amount_qbin'] = test_out['amount_qbin'].astype(str)
train_out.to_parquet(TRAIN_PATH, index=False)
test_out.to_parquet(TEST_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)

print('saved:', TRAIN_PATH)
print('saved:', TEST_PATH)
print('saved:', SUMMARY_PATH)

saved: ../../data/intermediate/df_train_stratified.parquet
saved: ../../data/intermediate/df_test_stratified.parquet
saved: ../../data/intermediate/df_train_test_stratified_summary.csv
